In [7]:
import pandas as pd
from itertools import product

# ----------------- CONFIGURACIÓN -----------------
FILE          = "data/sensibilidad-Copy1.csv"   # ruta al CSV
TARGET_SAFE   = 0.90    # umbral de éxito mínimo para la zona segura
TARGET_IDEAL  = 0.98    # umbral de éxito mínimo para la zona ideal
# -------------------------------------------------

# 1. Carga y limpieza -------------------------------------------
df = pd.read_csv(FILE)
# elimina cualquier columna cuyo nombre empiece por 'Unnamed'
df = df.loc[:, ~df.columns.str.contains(r"^Unnamed", case=False)]

# 2. Función para buscar la “mejor” zona ------------------------
def find_zone(data, target):
    """
    Devuelve un diccionario con los límites:
         ruido_max, periodo_min, frec_max, success, count
    que maximizan la cobertura manteniendo 'success' ≥ target.
    """
    ruido_vals   = sorted(data["ruido"].unique())
    periodo_vals = sorted(data["periodo"].unique())
    frec_vals    = sorted(data["frec_obs"].unique())

    best = None
    for r_max, p_min, f_max in product(ruido_vals, periodo_vals, frec_vals):
        subset = data[
            (data["ruido"]   <= r_max) &
            (data["periodo"] >= p_min) &
            (data["frec_obs"] <= f_max)
        ]
        if subset.empty:
            continue
        success = subset["ok"].mean()
        if success >= target:
            # prioriza subconjuntos con MÁS filas; si empatan, con ruido_max mayor
            if (best is None or
                len(subset) > best["count"] or
                (len(subset) == best["count"] and r_max > best["ruido_max"])):
                best = dict(
                    ruido_max   = r_max,
                    periodo_min = p_min,
                    frec_max    = f_max,
                    success     = success,
                    count       = len(subset)
                )
    return best

# 3. Cálculo de zonas -------------------------------------------
safe_zone  = find_zone(df, TARGET_SAFE)
ideal_zone = find_zone(df, TARGET_IDEAL)

# 4. Presentación -----------------------------------------------
def mostrar(nombre, zona):
    if zona is None:
        print(f"{nombre}: ningún rango alcanza el {(TARGET_SAFE if nombre=='SEGURA' else TARGET_IDEAL)*100:.0f}% de éxito.")
        return
    print(f"\n{nombre.center(40,'=')}")
    print(f"• ruido    ≤ {zona['ruido_max']}")
    print(f"• periodo  ≥ {zona['periodo_min']}")
    print(f"• frec_obs ≤ {zona['frec_max']}")
    print(f"→ éxito medio: {zona['success']*100:5.2f}%  "
          f"({zona['count']} combinaciones)")

mostrar("SEGURA", safe_zone)
mostrar("IDEAL",  ideal_zone)


=================SEGURA=================
• ruido    ≤ 3.7
• periodo  ≥ 91
• frec_obs ≤ 15
→ éxito medio: 90.09%  (999 combinaciones)

=================IDEAL==================
• ruido    ≤ 2.7
• periodo  ≥ 151
• frec_obs ≤ 8
→ éxito medio: 98.15%  (378 combinaciones)
